In [16]:
# kate c needed to run this before the first block to get it too work on her environment
import sys
!{sys.executable} -m pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [17]:
# example citations: 
# https://gist.github.com/cqtsma2/27af54ffddd6ababd114d9f642517611
# https://quantumnumbers.anu.edu.au/documentation
# https://2pisoftware.com/case-studies/anu-qrng/


import json
import requests
import time
from datetime import datetime, timedelta
import random
import pandas as pd

def get_quantum_numbers(api_key):
    url = "https://api.quantumnumbers.anu.edu.au"
    
    # my api key
    headers = {"x-api-key": api_key}
    # params becomes ?length=10&type=uint16
    params = {
        'length': 10,
        'type': 'uint16'
    }
    
    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        data = response.json()
        if data["success"] == True:
            return(data["data"])
        else:
            print(data["message"])
            return None
    
    else:
        print(f"Got an unexpected status-code: {response.status_code}")
        print(response.text)
        return None


In [18]:
def collect(api_key, total_requests, min_interval, max_interval):
    collected_data = []
    request_log = []
    # starting time for collection
    collection_start_time = datetime.now()

    for request in range(1, total_requests+1):
        # random time interval, between min and max, between this request and the next
        rand_time_interval = random.randint(min_interval, max_interval)

        #starting time of this request
        start = time.time()
        numbers = get_quantum_numbers(api_key)
        duration = time.time() - start
        
        # collecting metadata
        if numbers:
            collected_data.extend(numbers)
            success = True
            count = len(numbers)
        else: 
            success = False
            count = 0

        log_entry = {
            'request': request,
            'time_stamp': datetime.now().isoformat(),
            'duration (sec)': round(duration, 2),
            'success': success,
            'generated numbers count': count,
            'next_interval': rand_time_interval if request < total_requests else None # if last request, then None because we don't need it
        }
        request_log.append(log_entry)
        if request < total_requests:
            time.sleep(rand_time_interval)
    collection_end_time = datetime.now()
    total_duration = (collection_end_time - collection_start_time).total_seconds()
    return collected_data, request_log

In [19]:
# Usage (go to this website and create an account to get your key https://quantumnumbers.anu.edu.au/documentation)
API_KEY = "3mJ2uowr1d48qDTcOeZQP52LaDohbAbl4iUasG3O"
# amy's key: "gGIf3BaLAQ9Tc9PJpocEY5pCSD6GmwfK6fKzgwfX" collect(API_KEY, 100, 1, 20)
# kate's c key: "bvugIbVS5j51yPyZrpqv5gxBVRgA3G12HEBkOXY8" collect(API_KEY, 100, 1, 30)
# kate f's key: "jpLRcM3Eye9XG3hkkj1hB3tcEqQQMJ7K98yM0WXw" collect(API_KEY, 100, 1, 15)
# chloe's key: "57XsjvhCkW2GXxKnJs0ey3MuITiXtzFwfNSO4LZ1" collect(API_KEY, 100, 1, 25)
# james's key: "APzF1BUM731gkkM5NZ5oH6irvFHBPLpPBbfYn172" collect(API_KEY, 100, 1, 35)
# kate c key 2: "1mqC1EFcnlabKDgmkJuev1I9JfstWsHI6BluYLaL" collect(API_KEY, 100, 1, 25)
# kate c key 3: "3mJ2uowr1d48qDTcOeZQP52LaDohbAbl4iUasG3O" collect(API_KEY, 100, 1, 35)
# we want to vary the last varible (15, 25, 35) (20 and 30 already done)
collected_data, request_log = collect(API_KEY, 100, 1, 35)

In [20]:
df_log = pd.DataFrame(request_log)
print(df_log.head())

df_numbers = pd.DataFrame({
    'random_number': collected_data
})
print(df_numbers.head())

# numbers.csv = amy 6
# numbers_1.csv = kate c 5
# numbers_2.csv = kate f 7
# numbers_3.csv = chloe 
# numbers_4.csv = james
# kate key 2 = 8
# kate key 3 = 9
# change the number when you run it so you don't overwrite other data
df_log.to_csv('request_log_9.csv', index=False)
df_numbers.to_csv('numbers_9.csv', index=False)

   request                  time_stamp  duration (sec)  success  \
0        1  2026-05-02T15:37:02.774936            0.29     True   
1        2  2026-05-02T15:37:09.053283            0.28     True   
2        3  2026-05-02T15:37:12.308779            0.25     True   
3        4  2026-05-02T15:37:30.608214            0.30     True   
4        5  2026-05-02T15:37:45.868357            0.25     True   

   generated numbers count  next_interval  
0                       10            6.0  
1                       10            3.0  
2                       10           18.0  
3                       10           15.0  
4                       10            7.0  
   random_number
0          65431
1           6482
2           7698
3           3699
4          56417
